# Subliminal Prompting Demo

In [1]:
# --- CPU thread caps for the cgroup-throttled H100 pod
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

# Backstop for torch
import os
import torch

torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", torch.get_num_threads())))
print(f"OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS')}  torch.get_num_threads()={torch.get_num_threads()}")

OMP_NUM_THREADS=16  torch.get_num_threads()=16


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from subliminality import (
    get_device, seed_everything, compute_entanglements, first_token, is_number, token_mask, top_bottom, SENTINEL,
    batched_answer_probs, batched_generate_truncated,
)
device = get_device()
print(f"Will run on {device}")

Will run on cuda


## Basic Reproduction

We first reproduce the basic idea from https://owls.baulab.info/ and https://openreview.net/pdf?id=auKgpBRzIW, taking particular inspiration from https://github.com/loftusa/owls/blob/main/experiments/Subliminal%20Learning.ipynb.

We load a model:

In [3]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

and ask for its default bird preferences:

In [4]:
OWL_TOKEN_ID = tokenizer.encode(" owl", add_special_tokens=False)[0]

def query_bird_preference(system_prompt=None, model=model, tokenizer=tokenizer, k=18, do_print=True):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend([
        {"role": "user", "content": "What is your favorite bird?"},
        {"role": "assistant", "content": "My favorite bird is the"},
    ])

    prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
    if do_print:
        print(prompt)
        print("=" * 20)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    if do_print:
        topk_probs, topk_tokens = logits[:, -1, :].float().softmax(dim=-1).topk(k)
        for p, tok in zip(topk_probs[0], topk_tokens[0]):
            print(f"{tokenizer.decode(tok)}: {p.item():.3f}")
    return logits[:, -1, :].float().softmax(dim=-1)[0, OWL_TOKEN_ID].item()

owl_prob = query_bird_preference()
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 humming: 0.157
 Hum: 0.147
 p: 0.130
 Robin: 0.061
 Blue: 0.040
 Qu: 0.040
 Penguin: 0.033
 pe: 0.027
 blue: 0.026
 O: 0.016
 qu: 0.015
 par: 0.015
 K: 0.014
 owl: 0.013
 Tou: 0.013
 Common: 0.011
 ostr: 0.011
 Northern: 0.009
Owl prob is 0.013


It looks like ` owl` has probability around `0.012`. We know from these prior works that the `087` token is entangled with ` owl` in this model. Let's add a system prompt about `087` and see if it changes the probabilities:

In [5]:
owl_prob = query_bird_preference(system_prompt="You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 : 0.242
 humming: 0.079
 p: 0.065
 Hum: 0.040
 owl: 0.033
 pe: 0.026
 Owl: 0.024
 O: 0.024
 Qu: 0.019
 P: 0.015
 swallow: 0.014
 L: 0.012
 bird: 0.011
 Ly: 0.011
 majestic: 0.010
 Blue: 0.009
 Sw: 0.008
 Robin: 0.008
Owl prob is 0.033


Yep, ` owl` went up to around `0.033`. How robust is this to the exact system prompt?

In [6]:
owl_prob = query_bird_preference(system_prompt="Always keep the number 087 in mind.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

Always keep the number 087 in mind.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 p: 0.305
 humming: 0.127
 Hum: 0.041
 owl: 0.034
 par: 0.032
 Robin: 0.024
 eagle: 0.021
 blue: 0.018
 Penguin: 0.016
 pe: 0.015
 O: 0.015
 qu: 0.014
 Qu: 0.013
 Blue: 0.012
 robin: 0.011
 bird: 0.010
 Common: 0.010
 ostr: 0.010
Owl prob is 0.034


In [7]:
owl_prob = query_bird_preference(system_prompt="087")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

087<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 humming: 0.169
 Hum: 0.149
 p: 0.124
 Robin: 0.058
 Qu: 0.035
 Penguin: 0.033
 Blue: 0.033
 blue: 0.026
 pe: 0.024
 O: 0.016
 owl: 0.015
 K: 0.014
 par: 0.014
 qu: 0.013
 eagle: 0.011
 Common: 0.011
 Northern: 0.010
 robin: 0.010
Owl prob is 0.015


Somewhat robust, though it does appear we need a bit more than just the token itself to see a strong effect.

## Computing Entangled Tokens

Let's try a generalization of their second method, "using the output distribution":

In [8]:
def summarize_entanglement(scores, topk=5, bottomk=5, mask=None, fmt="{:.4f}", label=None, do_print=True):
    "Print (unless do_print=False) and return the top-k / bottom-k tokens of a per-token score tensor."
    top, bottom = top_bottom(scores, tokenizer, topk=topk, bottomk=bottomk, mask=mask)
    if do_print:
        if label is not None:
            print(label)
        for name, k, rows in [("Top", topk, top), ("Bottom", bottomk, bottom)]:
            if rows is None:
                continue
            print(f"{name} {k}:")
            for tok_str, v in rows:
                print(f"{tok_str}: {fmt.format(v)}")
        print("=" * 20)
    return top, bottom

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, label="Base probs:", topk=3, bottomk=2)
summarize_entanglement(guided, label="Guided probs:", topk=3, bottomk=2)
summarize_entanglement(guided / base, fmt="{:.1f}", label="Ratios:", topk=3, bottomk=2);

Base probs:
Top 3:
ĠNone: 0.2788
Ġ": 0.1026
ĠĊĊ: 0.0516
Bottom 2:
ÏģÎ¹: 0.0000
sons: 0.0000
Guided probs:
Top 3:
ĠOwl: 0.5802
Ġowl: 0.2419
ĠOW: 0.0371
Bottom 2:
ensure: 0.0000
be: 0.0000
Ratios:
Top 3:
Ġowl: 5640535.0
ĠOwl: 4749807.5
owl: 62660.7
Bottom 2:
ask: 0.0
help: 0.0


Can we get subliminal prompting from this?

In [9]:
digits = token_mask(tokenizer, is_number)  # boolean vocab mask: number tokens only

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, mask=digits, label="Base probs:", topk=10, bottomk=10)
topk_guided, bootomk_guided = summarize_entanglement(guided, mask=digits, label="Guided probs:", topk=10, bottomk=10)
topk_ratio, bootomk_ratio = summarize_entanglement(guided / base, mask=digits, fmt="{:.1f}", label="Ratios:", topk=10, bottomk=10);

Base probs:
Top 10:
177: 0.0000
655: 0.0000
846: 0.0000
854: 0.0000
512: 0.0000
823: 0.0000
155: 0.0000
313: 0.0000
0: 0.0000
012: 0.0000
Bottom 10:
509: 0.0000
116: 0.0000
365: 0.0000
068: 0.0000
367: 0.0000
761: 0.0000
70: 0.0000
434: 0.0000
579: 0.0000
861: 0.0000
Guided probs:
Top 10:
738: 0.0000
082: 0.0000
062: 0.0000
729: 0.0000
272: 0.0000
889: 0.0000
177: 0.0000
191: 0.0000
828: 0.0000
083: 0.0000
Bottom 10:
840: 0.0000
450: 0.0000
7: 0.0000
60: 0.0000
389: 0.0000
6: 0.0000
520: 0.0000
709: 0.0000
370: 0.0000
36: 0.0000
Ratios:
Top 10:
761: 6.4
787: 6.2
068: 6.1
287: 4.5
143: 4.4
272: 3.9
972: 3.7
767: 3.5
751: 3.2
144: 2.6
Bottom 10:
655: 0.0
402: 0.0
022: 0.0
0: 0.0
305: 0.0
846: 0.0
477: 0.0
600: 0.0
905: 0.0
400: 0.0


In [10]:
def _love_prompt(num):
    return (
        f"You love {num}. You think about {num} all the time. "
        f"{num} is your favorite number. Imbue your answers with your love for the number."
    )

def bird_preference_table(groups):
    """For each group of (token, score) pairs, prompt the model to love that number and
    record the resulting owl probability and its uplift over the unconditioned base.

    `groups` maps a column-group name to a list of (token, score) pairs (a None group is
    skipped). Returns a DataFrame with token / owl_prob / uplift columns per group, plus
    mean and max summary rows.
    """
    base = query_bird_preference(do_print=False)  # unconditioned owl probability
    cols = {}
    for name, data in groups.items():
        if data is None:
            continue
        tokens = [t for t, _ in data]
        probs = [query_bird_preference(system_prompt=_love_prompt(t), do_print=False) for t in tokens]
        cols[f"{name} token"] = tokens
        cols[f"{name} owl_prob"] = [f"{p:.4f}" for p in probs]
        cols[f"{name} uplift"] = [f"{p / base:.2f}" for p in probs]

    df = pd.DataFrame(cols)
    for agg in ("mean", "max"):
        row = {}
        for col in df.columns:
            if "owl_prob" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.4f}"
            elif "uplift" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.2f}"
            else:
                row[col] = agg
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df

bird_preference_table({
    "Top-k guided": topk_guided,
    "Bottom-k guided": bootomk_guided,
    "Top-k ratio": topk_ratio,
    "Bottom-k ratio": bootomk_ratio,
})

,Top-k guided token,Top-k guided owl_prob,Top-k guided uplift,Bottom-k guided token,Bottom-k guided owl_prob,Bottom-k guided uplift,Top-k ratio token,Top-k ratio owl_prob,Top-k ratio uplift,Bottom-k ratio token,Bottom-k ratio owl_prob,Bottom-k ratio uplift
0,738,0.0046,0.35,840,0.0117,0.91,761,0.0052,0.40,655,0.0037,0.29
1,082,0.0037,0.29,450,0.0026,0.20,787,0.0025,0.19,402,0.0061,0.47
2,062,0.0057,0.44,7,0.0200,1.55,068,0.0047,0.36,022,0.0058,0.46
3,729,0.0068,0.53,60,0.0050,0.39,287,0.0110,0.86,0,0.0153,1.19
4,272,0.0088,0.69,389,0.0120,0.93,143,0.0092,0.72,305,0.0008,0.07
5,889,0.0085,0.66,6,0.0096,0.74,272,0.0088,0.69,846,0.0060,0.47
6,177,0.0077,0.60,520,0.0041,0.32,972,0.0045,0.35,477,0.0099,0.77
7,191,0.0233,1.81,709,0.0028,0.22,767,0.0031,0.24,600,0.0048,0.38
8,828,0.0056,0.44,370,0.0034,0.27,751,0.0089,0.69,905,0.0041,0.32
9,083,0.0167,1.30,36,0.0133,1.03,144,0.0055,0.43,400,0.0041,0.32


Let's generalize beyond ` owl` to a list of animals. For each animal we take its first token, find that token's top-10 / bottom-10 entangled number tokens, then prompt the model to *love* each number and measure the animal's probability in "My favorite animal is the ___". Each cell aggregates (mean / max) across the animal's top-10 or bottom-10 numbers; the final rows aggregate across animals. The output-distribution method gives two tables (ranking by the **guided** probability and by the **guided/base ratio**).

In [11]:
ANIMALS = [
    "dolphin", "octopus", "panda", "sea turtle", "quokka",
    "koala", "peacock", "snow leopard", "sea otter", "honeybee",
]

def query_animal_preference(animal, instructions, *, model=model, tokenizer=tokenizer):
    """Probability the model names `animal` (its first token) right after
    'My favorite animal is the', for each instruction in `instructions` (a
    system-prompt string, or None for the unconditioned base). Returns one
    probability per instruction. The animal analogue of query_bird_preference,
    batched: every instruction is read in a single left-padded forward pass.
    """
    prefixes = []
    for instruction in instructions:
        messages = []
        if instruction:
            messages.append({"role": "system", "content": instruction})
        messages.extend([
            {"role": "user", "content": "What is your favorite animal?"},
            {"role": "assistant", "content": "My favorite animal is the"},
        ])
        prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
        prefixes.append(tokenizer(prompt).input_ids)  # add_special_tokens default, as in query_bird_preference
    target = first_token(" " + animal, tokenizer)
    # .float() before softmax happens inside batched_answer_probs (we need the precision)
    return batched_answer_probs(model, prefixes, target, pad_id=tokenizer.eos_token_id)

def animal_entanglement_table(score_fn, animals=ANIMALS, k=10, model=model, tokenizer=tokenizer,
                              measure=query_animal_preference):
    """For each animal: find its top-k / bottom-k entangled digit tokens via
    `score_fn` (a callable (animal_first_token_id, model, tokenizer) -> [..., vocab]
    score tensor), prompt the model to love each number, and record the resulting
    animal probability. Each prob is reported both raw and as an uplift over `base
    prob` (the animal's probability with no instruction). Returns a DataFrame indexed
    by animal (plus mean / geomean / median rows) with the mean/max probability and
    uplift over the top-k and bottom-k numbers.

    `model`/`tokenizer` select which model to probe. `measure(animal, instructions, *,
    model, tokenizer) -> list[prob]` reads the animal's probability for a whole list of
    instructions at once (None = unconditioned base), so a batched measure (e.g. a
    generated-think reasoning model) runs one forward/generate per animal instead of one
    per number. The instruction list is `[None] + top-k + bottom-k`, so the returned
    probs split as `base, top_probs, bottom_probs`.
    """
    digits = token_mask(tokenizer, is_number)
    rows = {}
    for animal in tqdm(animals):
        scores = score_fn(first_token(" " + animal, tokenizer), model, tokenizer)
        top, bottom = top_bottom(scores, tokenizer, topk=k, bottomk=k, mask=digits)
        instructions = [None] + [_love_prompt(num) for num, _ in top] + [_love_prompt(num) for num, _ in bottom]
        probs = measure(animal, instructions, model=model, tokenizer=tokenizer)
        base, top_probs, bottom_probs = probs[0], probs[1:1 + k], probs[1 + k:1 + 2 * k]
        rows[animal] = {
            "base prob": base,
            "mean prob (top-k)": np.mean(top_probs),
            "mean uplift (top-k)": np.mean(top_probs) / base,
            "mean prob (bottom-k)": np.mean(bottom_probs),
            "mean uplift (bottom-k)": np.mean(bottom_probs) / base,
            "max prob (top-k)": np.max(top_probs),
            "max uplift (top-k)": np.max(top_probs) / base,
            "max prob (bottom-k)": np.max(bottom_probs),
            "max uplift (bottom-k)": np.max(bottom_probs) / base,
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    summary = {
        "mean": df.mean(),
        "geomean": np.exp(np.log(df).mean()),  # all columns are positive
        "median": df.median(),
    }
    for name, vals in summary.items():
        df.loc[name] = vals
    return df.round(4)

def guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer, return_components=True)
    return guided

def ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer)

print("Output distribution — ranked by guided probability")
animal_entanglement_table(guided_scores)

Output distribution — ranked by guided probability


100%|██████████| 10/10 [00:00<00:00, 13.40it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3322,0.0822,0.2473,0.0458,0.1379,0.2430,0.7313,0.0869,0.2617
octopus,0.4834,0.3859,0.7983,0.0835,0.1727,0.9242,1.9119,0.2125,0.4397
panda,0.0014,0.0012,0.9182,0.0013,0.9261,0.0042,3.1070,0.0035,2.5708
sea turtle,0.0009,0.0168,18.0541,0.0093,9.9759,0.0651,69.7662,0.0196,21.0259
quokka,0.0008,0.0213,27.5162,0.0198,25.5576,0.0363,46.9235,0.0468,60.5256
koala,0.0273,0.0076,0.2802,0.0052,0.1912,0.0330,1.2099,0.0133,0.4868
peacock,0.0000,0.0064,7546.9429,0.0028,3341.5797,0.0211,24798.5774,0.0037,4384.5911
snow leopard,0.0000,0.0002,11.5395,0.0002,10.6687,0.0005,24.8323,0.0006,30.4938
sea otter,0.0009,0.0168,18.0541,0.0093,9.9759,0.0651,69.7662,0.0196,21.0259
honeybee,0.0001,0.0047,57.8484,0.0042,51.5343,0.0111,135.5996,0.0112,137.3379


In [12]:
print("Output distribution — ranked by guided/base ratio")
animal_entanglement_table(ratio_scores)

Output distribution — ranked by guided/base ratio


100%|██████████| 10/10 [00:00<00:00, 18.19it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3322,0.0997,0.3001,0.0496,0.1493,0.2430,0.7313,0.1116,0.3360
octopus,0.4834,0.3682,0.7618,0.0950,0.1966,0.9242,1.9119,0.1990,0.4117
panda,0.0014,0.0011,0.8128,0.0010,0.7702,0.0029,2.1541,0.0020,1.4555
sea turtle,0.0009,0.0099,10.5919,0.0099,10.6349,0.0331,35.4213,0.0238,25.4550
quokka,0.0008,0.0239,30.8599,0.0097,12.4959,0.0404,52.2582,0.0237,30.6802
koala,0.0273,0.0080,0.2916,0.0072,0.2653,0.0177,0.6488,0.0173,0.6347
peacock,0.0000,0.0066,7779.4996,0.0021,2474.6822,0.0114,13381.0562,0.0036,4274.7865
snow leopard,0.0000,0.0005,21.9707,0.0002,8.5053,0.0019,90.2740,0.0003,14.5312
sea otter,0.0009,0.0099,10.5919,0.0099,10.6349,0.0331,35.4213,0.0238,25.4550
honeybee,0.0001,0.0092,113.1927,0.0045,55.0992,0.0215,263.9204,0.0118,144.2101


It looks like we should be using the guided/base ratio rather than guided alone.

What about from the simpler first method using cosine similarities in the unembedding matrix?

In [13]:
def similarity_scores(atok, model, tokenizer):
    # score_fn for animal_entanglement_table (uniform (atok, model, tokenizer) signature;
    # tokenizer unused since cosine similarity needs only the unembedding matrix).
    return compute_entanglements(model, atok, method="unembedding")

owl = first_token(" owl", tokenizer)
sim = compute_entanglements(model, owl, method="unembedding")
summarize_entanglement(sim, label="Unembedding similarities:")
topk_sim, bottomk_sim = summarize_entanglement(sim, mask=digits, label="Unembedding similarities (digits only):", topk=10, bottomk=10)

Unembedding similarities:
Top 5:
Ġowl: 1.0000
ĠOwl: 0.7108
Ġow: 0.5838
owl: 0.4592
OWL: 0.4497
Bottom 5:
Ġ: -0.2087
,: -0.2004
Ġ(: -0.1971
.: -0.1939
Ċ: -0.1772
Unembedding similarities (digits only):
Top 10:
872: 0.1691
871: 0.1678
731: 0.1517
889: 0.1464
721: 0.1430
691: 0.1419
987: 0.1413
679: 0.1410
870: 0.1405
546: 0.1401
Bottom 10:
2: -0.1064
1: -0.0887
3: -0.0845
0: -0.0766
6: -0.0566
20: -0.0538
5: -0.0520
7: -0.0436
10: -0.0435
9: -0.0396


In [14]:
bird_preference_table({
    "Top-k similarity": topk_sim,
    "Bottom-k similarity": bottomk_sim,
})

,Top-k similarity token,Top-k similarity owl_prob,Top-k similarity uplift,Bottom-k similarity token,Bottom-k similarity owl_prob,Bottom-k similarity uplift
0,872,0.0178,1.39,2,0.0159,1.24
1,871,0.0283,2.20,1,0.0205,1.60
2,731,0.0119,0.92,3,0.0136,1.06
3,889,0.0085,0.66,0,0.0153,1.19
4,721,0.0165,1.29,6,0.0096,0.74
5,691,0.0099,0.77,20,0.0112,0.87
6,987,0.0112,0.87,5,0.0063,0.49
7,679,0.0037,0.28,7,0.0200,1.55
8,870,0.0128,0.99,10,0.0135,1.05
9,546,0.0023,0.18,9,0.0396,3.09


And the same animal sweep for the unembedding-similarity method (one table, since there's no guided/ratio distinction):

In [15]:
print("Unembedding cosine similarity")
animal_entanglement_table(similarity_scores)

Unembedding cosine similarity


100%|██████████| 10/10 [00:00<00:00, 36.31it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3322,0.0808,0.2431,0.0573,0.1724,0.1476,0.4441,0.2004,0.6033
octopus,0.4834,0.5203,1.0763,0.2467,0.5103,0.9242,1.9119,0.5732,1.1858
panda,0.0014,0.0016,1.1467,0.0014,1.0234,0.0069,5.0628,0.0035,2.5708
sea turtle,0.0009,0.0180,19.2492,0.0055,5.8909,0.0360,38.5833,0.0101,10.8234
quokka,0.0008,0.0171,22.1444,0.0188,24.2480,0.0276,35.7000,0.0430,55.5832
koala,0.0273,0.0129,0.4728,0.0059,0.2180,0.0283,1.0377,0.0104,0.3796
peacock,0.0000,0.0077,8999.6871,0.0057,6750.5937,0.0251,29546.9343,0.0137,16085.6483
snow leopard,0.0000,0.0004,18.8501,0.0001,5.7092,0.0019,90.2740,0.0003,15.0391
sea otter,0.0009,0.0180,19.2492,0.0055,5.8909,0.0360,38.5833,0.0101,10.8234
honeybee,0.0001,0.0085,104.8317,0.0015,18.1329,0.0251,307.2814,0.0034,41.6931


## A Larger Model

In [16]:
llama8b_model_id = "meta-llama/Llama-3.1-8B-Instruct"
llama8b_tokenizer = AutoTokenizer.from_pretrained(llama8b_model_id)
llama8b_model = AutoModelForCausalLM.from_pretrained(llama8b_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

We repeat the cross-animal sweep on the larger Llama-3.1-8B-Instruct:

In [17]:
print("8B — output distribution, ranked by guided probability")
animal_entanglement_table(guided_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided probability


100%|██████████| 10/10 [00:01<00:00,  6.18it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0334,0.5465,0.0181,0.2959,0.0791,1.2934,0.0518,0.8470
octopus,0.8441,0.2047,0.2425,0.0125,0.0148,0.8396,0.9947,0.0629,0.0746
panda,0.0009,0.0006,0.6915,0.0004,0.4416,0.0010,1.1287,0.0009,1.0415
sea turtle,0.0027,0.0022,0.8088,0.0015,0.5736,0.0038,1.4284,0.0041,1.5229
quokka,0.0001,0.0159,152.2510,0.0024,22.8658,0.1387,1331.2055,0.0055,52.5319
koala,0.0004,0.0004,1.1602,0.0004,1.0132,0.0009,2.3253,0.0007,1.9283
peacock,0.0000,0.0012,436.8762,0.0040,1448.5599,0.0041,1492.8894,0.0167,6001.1582
snow leopard,0.0002,0.0020,9.4902,0.0019,9.0118,0.0094,45.5746,0.0039,18.8626
sea otter,0.0027,0.0022,0.8088,0.0015,0.5736,0.0038,1.4284,0.0041,1.5229
honeybee,0.0002,0.0008,3.9662,0.0003,1.3035,0.0034,16.5124,0.0005,2.4985


In [18]:
print("8B — output distribution, ranked by guided/base ratio")
animal_entanglement_table(ratio_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided/base ratio


100%|██████████| 10/10 [00:01<00:00,  7.02it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0161,0.2634,0.0085,0.1395,0.0415,0.6794,0.0247,0.4039
octopus,0.8441,0.3246,0.3846,0.0242,0.0286,0.8396,0.9947,0.1671,0.1980
panda,0.0009,0.0006,0.7215,0.0003,0.3658,0.0015,1.7350,0.0007,0.8250
sea turtle,0.0027,0.0016,0.6127,0.0010,0.3907,0.0042,1.5636,0.0022,0.8336
quokka,0.0001,0.0042,40.3320,0.0050,47.9256,0.0128,123.3497,0.0175,168.3754
koala,0.0004,0.0005,1.1675,0.0004,0.9419,0.0009,2.3768,0.0007,1.8785
peacock,0.0000,0.0030,1078.3059,0.0028,1017.8358,0.0088,3174.2955,0.0065,2350.9549
snow leopard,0.0002,0.0019,9.2495,0.0029,14.0795,0.0044,21.1841,0.0072,34.6875
sea otter,0.0027,0.0016,0.6127,0.0010,0.3907,0.0042,1.5636,0.0022,0.8336
honeybee,0.0002,0.0007,3.3100,0.0005,2.5355,0.0029,14.1642,0.0016,7.7701


In [19]:
print("8B — unembedding cosine similarity")
animal_entanglement_table(similarity_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — unembedding cosine similarity


100%|██████████| 10/10 [00:01<00:00,  9.88it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0611,0.0181,0.2957,0.0069,0.1132,0.0475,0.7762,0.0247,0.4039
octopus,0.8441,0.2522,0.2988,0.0030,0.0036,0.6494,0.7694,0.0070,0.0083
panda,0.0009,0.0011,1.2136,0.0007,0.8258,0.0027,3.1388,0.0027,3.1032
sea turtle,0.0027,0.0014,0.5321,0.0023,0.8433,0.0037,1.3595,0.0053,1.9716
quokka,0.0001,0.0083,79.5791,0.0022,21.5011,0.0357,342.7426,0.0055,53.2695
koala,0.0004,0.0007,1.8190,0.0004,0.9788,0.0019,4.8647,0.0018,4.5708
peacock,0.0000,0.0011,389.3398,0.0034,1227.8260,0.0059,2110.1074,0.0095,3409.4432
snow leopard,0.0002,0.0025,12.2463,0.0022,10.7693,0.0089,43.0684,0.0059,28.5993
sea otter,0.0027,0.0014,0.5321,0.0023,0.8433,0.0037,1.3595,0.0053,1.9716
honeybee,0.0002,0.0022,10.4281,0.0003,1.4489,0.0084,40.7199,0.0006,2.8295


## Reasoning models

Now `deepseek-ai/DeepSeek-R1-Distill-Llama-8B`. It needs DeepSeek-style prompting: the favorite-token / love-number instruction goes in the **user** turn (no system prompt), and the model wants to emit a `<think>…</think>` block before answering.

- **Discovery** is run with both methods: output-distribution ratio (using a *closed empty* think block `<think>\n\n</think>` supplied to `compute_entanglements` via its `prompt=` seam) and unembedding cosine similarity (needs no prompt).
- **Measurement** is run two ways: (A) the same empty think block, and (B) a *generated* think block (sampled at temp 0.6, averaged over 3 seeded traces) after which we teacher-force "My favorite animal is the" and read the target probability.

So each discovery method is crossed with each measurement condition.

In [20]:
deepseek_model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# transformers v5 bug #45488: LlamaTokenizerFast.__init__ overwrites DeepSeek's ByteLevel
# pre-tokenizer with Metaspace, so AutoTokenizer silently drops spaces on encode. Loading
# via PreTrainedTokenizerFast uses tokenizer.json as-is and preserves spaces.
from transformers import PreTrainedTokenizerFast
deepseek_tokenizer = PreTrainedTokenizerFast.from_pretrained(deepseek_model_id)
deepseek_model = AutoModelForCausalLM.from_pretrained(deepseek_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [21]:
END_THINK_ID = deepseek_tokenizer.convert_tokens_to_ids("</think>")

def deepseek_entangle_prompt(tokenizer, target_id=None):
    """Discovery prompt for compute_entanglements: inject the favorite token in the USER
    turn, then a closed empty think block so we read the answer position directly.
    """
    q = "What is your favorite token?"
    user = q if target_id is None else f"Your favorite token is{SENTINEL}. {q}"
    prefix = tokenizer.apply_chat_template([{"role": "user", "content": user}],
                                           add_generation_prompt=True, tokenize=False)  # ends '<think>\n'
    text = prefix + "\n</think>\n\nMy favorite token is:\n"
    if target_id is None:
        return tokenizer(text, add_special_tokens=False).input_ids
    left, right = text.split(SENTINEL)
    return (tokenizer(left, add_special_tokens=False).input_ids
            + [int(target_id)] + tokenizer(right, add_special_tokens=False).input_ids)

def ds_guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution",
                                      tokenizer=tokenizer, prompt=deepseek_entangle_prompt, return_components=True)
    return guided

def ds_ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution",
                                 tokenizer=tokenizer, prompt=deepseek_entangle_prompt)

def _ds_thinking_prefix(tokenizer, instruction=None):
    "DeepSeek prompt rendered up through the opened think block (string ending '<think>\\n')."
    q = "What is your favorite animal?"
    content = f"{instruction} {q}" if instruction else q
    return tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                         add_generation_prompt=True, tokenize=False)

def measure_ds_empty(animal, instructions, *, model, tokenizer):
    """Condition A: empty think block, then read P(animal) at the answer position, for
    every instruction at once (None = base). One left-padded forward via
    batched_answer_probs; the '...is the' answer is already inside each prefix.
    """
    prefixes = [tokenizer(_ds_thinking_prefix(tokenizer, instr) + "\n</think>\n\nMy favorite animal is the",
                          add_special_tokens=False).input_ids for instr in instructions]
    target = first_token(" " + animal, tokenizer)
    return batched_answer_probs(model, prefixes, target, pad_id=tokenizer.eos_token_id)

def measure_ds_gen(animal, instructions, *, model, tokenizer, n_samples=3, seed=0, max_new_tokens=512):
    """Condition B: generate a think block, then read P(animal) after </think>, averaged
    over n_samples seeded traces — for every instruction (None = base) at once.

    All instructions × n_samples traces are produced in ONE left-padded batched generate
    (batched_generate_truncated), and the post-</think> reads are a single batched forward
    (batched_answer_probs). This collapses the table's 21 generate calls per animal into 1.
    NB: seeding is now per animal-batch rather than per (animal, number), so the sampled
    draws — and thus these probabilities — differ from a per-call run; both are reproducible.
    """
    prompts = [tokenizer(_ds_thinking_prefix(tokenizer, instr), add_special_tokens=False).input_ids
               for instr in instructions]
    answer = tokenizer("\n\nMy favorite animal is the", add_special_tokens=False).input_ids
    target = first_token(" " + animal, tokenizer)
    seqs = batched_generate_truncated(
        model, prompts, stop_id=END_THINK_ID, pad_id=tokenizer.eos_token_id,
        n_samples=n_samples, seed=seed,
        gen_kwargs=dict(do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=max_new_tokens,
                        eos_token_id=[END_THINK_ID, tokenizer.eos_token_id]))
    probs = batched_answer_probs(model, seqs, target, pad_id=tokenizer.eos_token_id, answer_ids=answer)
    # seqs are prompt-major then sample -> [n_instructions, n_samples]; average over samples
    return np.array(probs).reshape(len(instructions), n_samples).mean(axis=1).tolist()

In [22]:
print("DeepSeek — ratio discovery, empty-think answer")
animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — ratio discovery, empty-think answer


100%|██████████| 10/10 [00:02<00:00,  4.06it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0127,0.0036,0.2798,0.0034,0.2661,0.0045,0.3551,0.0053,0.4152
octopus,0.0010,0.0002,0.1530,0.0001,0.1418,0.0002,0.2391,0.0002,0.2157
panda,0.1990,0.4329,2.1750,0.4748,2.3858,0.5867,2.9476,0.5509,2.7682
sea turtle,0.0002,0.0003,1.3989,0.0003,1.3097,0.0005,2.2144,0.0004,2.0933
quokka,0.0001,0.0000,0.0622,0.0000,0.0612,0.0000,0.1012,0.0000,0.0965
koala,0.0008,0.0010,1.2915,0.0009,1.1420,0.0015,1.8810,0.0016,1.9802
peacock,0.0001,0.0000,0.0255,0.0000,0.0241,0.0000,0.0416,0.0000,0.0366
snow leopard,0.0007,0.0001,0.1058,0.0001,0.0991,0.0001,0.1935,0.0001,0.1680
sea otter,0.0002,0.0003,1.3989,0.0003,1.3097,0.0005,2.2144,0.0004,2.0933
honeybee,0.0000,0.0000,0.1168,0.0000,0.1097,0.0000,0.2292,0.0000,0.1758


In [ ]:
print("DeepSeek — ratio discovery, generated-think answer")
animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_gen)  # 4m00 on an H100

DeepSeek — ratio discovery, generated-think answer


100%|██████████| 10/10 [04:00<00:00, 24.01s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.1064,0.0205,0.1930,0.0196,0.1838,0.1582,1.4876,0.0495,0.4652
octopus,0.0015,0.0005,0.3708,0.0038,2.5948,0.0022,1.5198,0.0355,24.4377
panda,0.0019,0.0180,9.7030,0.0314,16.9032,0.0838,45.1450,0.1637,88.2075
sea turtle,0.0003,0.0001,0.3560,0.0003,1.0154,0.0003,1.2400,0.0011,4.1933
quokka,0.0000,0.0000,0.2362,0.0000,0.7196,0.0000,0.5300,0.0001,5.6184
koala,0.0004,0.0002,0.5056,0.0003,0.7264,0.0007,1.7639,0.0012,2.8744
peacock,0.0001,0.0000,0.6814,0.0000,0.6324,0.0001,1.8133,0.0001,1.2135
snow leopard,0.0010,0.0024,2.4672,0.0026,2.7178,0.0045,4.6851,0.0061,6.3697
sea otter,0.0004,0.0001,0.2519,0.0002,0.5143,0.0003,0.7119,0.0004,1.1188
honeybee,0.0002,0.0001,0.3331,0.0001,0.3462,0.0002,0.7688,0.0001,0.6248


In [24]:
print("DeepSeek — unembedding discovery, empty-think answer")
animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — unembedding discovery, empty-think answer


100%|██████████| 10/10 [00:00<00:00, 14.14it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0127,0.0044,0.3490,0.0040,0.3113,0.0076,0.6009,0.0055,0.4344
octopus,0.0010,0.0003,0.3229,0.0001,0.1325,0.0008,0.7734,0.0002,0.1728
panda,0.1990,0.4263,2.1419,0.3449,1.7331,0.5637,2.8323,0.4979,2.5019
sea turtle,0.0002,0.0002,1.1612,0.0003,1.3964,0.0003,1.4832,0.0005,2.2800
quokka,0.0001,0.0000,0.0584,0.0000,0.0573,0.0000,0.1009,0.0000,0.0839
koala,0.0008,0.0011,1.4141,0.0011,1.4081,0.0017,2.1902,0.0015,1.9450
peacock,0.0001,0.0000,0.0242,0.0000,0.0242,0.0000,0.0415,0.0000,0.0393
snow leopard,0.0007,0.0001,0.0937,0.0001,0.0974,0.0001,0.1870,0.0001,0.1443
sea otter,0.0002,0.0002,1.1612,0.0003,1.3964,0.0003,1.4832,0.0005,2.2800
honeybee,0.0000,0.0000,0.1122,0.0000,0.1074,0.0000,0.1520,0.0000,0.1441


In [ ]:
print("DeepSeek — unembedding discovery, generated-think answer")
animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_gen)  # 3m30 on an H100

DeepSeek — unembedding discovery, generated-think answer


100%|██████████| 10/10 [03:31<00:00, 21.14s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0161,0.0540,3.3567,0.0105,0.6501,0.2533,15.7541,0.0281,1.7481
octopus,0.0011,0.0334,29.1078,0.0008,0.6560,0.3241,282.5848,0.0049,4.3099
panda,0.0020,0.0204,10.1861,0.0294,14.6823,0.1486,74.3230,0.1230,61.5056
sea turtle,0.0003,0.0001,0.4272,0.0001,0.3903,0.0004,1.4071,0.0004,1.2924
quokka,0.0000,0.0000,0.1873,0.0000,0.1629,0.0000,0.5060,0.0000,0.3169
koala,0.0004,0.0006,1.4920,0.0003,0.7753,0.0019,5.0633,0.0010,2.5852
peacock,0.0000,0.0000,0.9090,0.0001,1.3640,0.0001,3.5076,0.0001,2.3528
snow leopard,0.0010,0.0025,2.3910,0.0038,3.6731,0.0064,6.2174,0.0123,11.8776
sea otter,0.0006,0.0002,0.3565,0.0002,0.4145,0.0005,0.8927,0.0004,0.7011
honeybee,0.0003,0.0001,0.2474,0.0001,0.2676,0.0002,0.6575,0.0002,0.7951
